In [2]:
import argparse
import pandas as pd
import numpy as np
import os

from scipy.stats import shapiro, mannwhitneyu, ttest_ind
from statsmodels.stats.multitest import multipletests
from sklearn import metrics

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score, precision_score,
                             recall_score, balanced_accuracy_score, cohen_kappa_score,
                             matthews_corrcoef, confusion_matrix)
from sklearn.feature_selection import VarianceThreshold

import matplotlib.pyplot as plt
import seaborn as sns

import matplotlib as mpl
from sklearn.preprocessing import label_binarize
mpl.use('Agg')
import scienceplots

plt.style.use(['science', 'grid'])
dpi = 300
from scipy.stats import kruskal, f_oneway
plt.rcParams["text.usetex"] = False


In [2]:
def get_models(random_state=42):
    """
    Define los pipelines para cada clasificador, incluyendo preprocesamiento estándar.
    
    Args:
        random_state (int): Semilla para reproducibilidad
    
    Returns:
        list: Lista de tuplas (nombre_modelo, pipeline_scikit)
    """

    # Pipeline para Support Vector Machine
    pipe_svc = make_pipeline(
        StandardScaler(), # Normalización de características
        VarianceThreshold(),  # Eliminación de características con varianza nula
        SVC(random_state=random_state, class_weight="balanced", probability=True)
    )
    
    # Pipeline para Regresión Logística
    pipe_lr = make_pipeline(
        StandardScaler(),
        VarianceThreshold(),
        LogisticRegression(
            penalty='elasticnet',       # Regularización combinada L1 y L2
            l1_ratio=0.5,               # Ratio para elasticnet (0.5 = igual peso L1 y L2)
            class_weight="balanced",
            random_state=random_state,
            solver='saga',              # Optimizador para elasticnet
            max_iter=10000              # Iteraciones máximas
        )
    )
    
    # Pipeline para Random Forest
    pipe_rf = make_pipeline(
        StandardScaler(),
        VarianceThreshold(),
        RandomForestClassifier(n_jobs=-1, class_weight="balanced_subsample", random_state=random_state)
    )
    
    # Pipeline para Naive Bayes Gaussiano
    pipe_nb = make_pipeline(
        StandardScaler(),
        VarianceThreshold(),
        GaussianNB() # No necesita parámetros adicionales
    )
    
    # Pipeline para K-Nearest Neighbors
    pipe_knn = make_pipeline(
        StandardScaler(),
        VarianceThreshold(),
        KNeighborsClassifier(n_jobs=-1)
    )
    
    # Pipeline para Gradient Boosting
    pipe_gb = make_pipeline(
        StandardScaler(),
        VarianceThreshold(),
        GradientBoostingClassifier(random_state=random_state)
    )

    # Lista con todos los modelos
    models = [
        ("SVM", pipe_svc),
        ("Logistic Regression", pipe_lr),
        ("Random Forest", pipe_rf),
        ("Naive Bayes", pipe_nb),
        ("KNN", pipe_knn),
        ("Gradient Boosting", pipe_gb),
    ]
    return models

# Multiclass

In [1]:
def evaluate_model_multiclass(model, X, y, groups, n_splits=5, n_repeats=1, base_random_state=42):
    """
    Realiza validación cruzada repetida estratificada por grupos (multiclase).
    
    Args:
        model: Modelo a evaluar (pipeline de scikit-learn)
        X (pd.DataFrame): Características
        y (np.array): Etiquetas multiclase
        groups (np.array): Identificadores de grupos (pacientes) para CV
        n_splits (int): Número de particiones por repetición
        n_repeats (int): Número de repeticiones de la validación cruzada
        base_random_state (int): Semilla base para reproducibilidad
    
    Returns:
        tuple: (fold_results, pred_vals)
            - fold_results: Lista de diccionarios con métricas por fold
            - pred_vals: Dict con datos de predicciones para cada fold
    """
    fold_results = []
    folds_data = []
    global_fold_index = 0
    classes = np.unique(y)
    for rep in range(n_repeats):
        current_random_state = base_random_state + rep
        splitter = StratifiedGroupKFold(
            n_splits=n_splits, shuffle=True, random_state=current_random_state
        )
        for train_idx, val_idx in splitter.split(X, y, groups=groups):
            global_fold_index += 1
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            model.fit(X_train, y_train)
            y_train_pred = model.predict(X_train)

            # Probabilidades o scores
            if hasattr(model, "predict_proba"):
                y_train_prob = model.predict_proba(X_train)
            elif hasattr(model, "decision_function"):
                y_train_prob = model.decision_function(X_train)
            else:
                y_train_prob = None

            # AUC multiclase en entrenamiento
            try:
                y_train_bin = label_binarize(y_train, classes=classes)
                if y_train_prob is not None and len(np.unique(y_train)) > 1:
                    train_auc = roc_auc_score(y_train_bin, y_train_prob, multi_class="ovr", average="macro")
                else:
                    train_auc = np.nan
            except:
                train_auc = np.nan

            train_f1_macro = f1_score(y_train, y_train_pred, average="macro")

            # Validación
            y_val_pred = model.predict(X_val)
            if hasattr(model, "predict_proba"):
                y_val_prob = model.predict_proba(X_val)
            elif hasattr(model, "decision_function"):
                y_val_prob = model.decision_function(X_val)
            else:
                y_val_prob = None

            # AUC multiclase en validación
            try:
                y_val_bin = label_binarize(y_val, classes=classes)
                if y_val_prob is not None and len(np.unique(y_val)) > 1:
                    val_auc = roc_auc_score(y_val_bin, y_val_prob, multi_class="ovr", average="macro")
                else:
                    val_auc = np.nan
            except:
                val_auc = np.nan

            val_mcc = matthews_corrcoef(y_val, y_val_pred)
            val_kappa = cohen_kappa_score(y_val, y_val_pred)
            val_f1_macro = f1_score(y_val, y_val_pred, average="macro")
            val_accuracy = accuracy_score(y_val, y_val_pred)
            val_balanced_accuracy = balanced_accuracy_score(y_val, y_val_pred)

            # Métricas por clase
            per_class_precision = precision_score(y_val, y_val_pred, average=None, labels=classes)
            per_class_recall = recall_score(y_val, y_val_pred, average=None, labels=classes)
            per_class_f1 = f1_score(y_val, y_val_pred, average=None, labels=classes)

            # Matriz de confusión y exactitud por clase
            cm = confusion_matrix(y_val, y_val_pred, labels=classes)
            per_class_accuracy = []
            for i in range(len(cm)):
                row_sum = np.sum(cm[i, :])
                if row_sum > 0:
                    per_class_accuracy.append(cm[i, i] / row_sum)
                else:
                    per_class_accuracy.append(np.nan)

            fold_metrics = {
                "Fold": global_fold_index,
                "Repeat": rep + 1,
                "train_auc": train_auc,
                "train_f1_macro": train_f1_macro,
                "val_auc": val_auc,
                "val_mcc": val_mcc,
                "val_kappa": val_kappa,
                "val_f1_macro": val_f1_macro,
                "val_accuracy": val_accuracy,
                "val_balanced_accuracy": val_balanced_accuracy,
                "per_class_precision": per_class_precision.tolist(),
                "per_class_recall": per_class_recall.tolist(),
                "per_class_f1": per_class_f1.tolist(),
                "per_class_accuracy": per_class_accuracy
            }
            fold_results.append(fold_metrics)

            folds_data.append({
                "fold_index": global_fold_index,
                "Repeat": rep + 1,
                "y_val": y_val,
                "y_val_pred": y_val_pred,
                "y_val_prob": y_val_prob
            })

    pred_vals = {
        "folds": folds_data
    }
    return fold_results, pred_vals

# Binario

In [4]:
def evaluate_model_binary(model, X, y, groups, n_splits=5, n_repeats=1, base_random_state=42):
    """
    Realiza validación cruzada repetida estratificada por grupos (pacientes).
    
    Args:
        model: Modelo a evaluar (pipeline de scikit-learn)
        X (pd.DataFrame): Características
        y (np.array): Etiquetas binarias (0/1)
        groups (np.array): Identificadores de grupos (pacientes) para CV
        n_splits (int): Número de particiones por repetición
        n_repeats (int): Número de repeticiones de la validación cruzada
        base_random_state (int): Semilla base para reproducibilidad
    
    Returns:
        tuple: (fold_results, pred_vals)
            - fold_results: Lista de diccionarios con métricas por fold
            - pred_vals: Dict con datos de predicciones para cada fold
    """

    fold_results = []   # Lista para almacenar métricas de cada fold
    folds_data = []     # Lista para almacenar datos de predicciones

    global_fold_index = 0
    for rep in range(n_repeats):
        # Cada repetición usa una semilla diferente para obtener distintas particiones
        current_random_state = base_random_state + rep
        
        # StratifiedGroupKFold garantiza distribución similar de clases
        # manteniendo separación de grupos (pacientes) entre train/val
        splitter = StratifiedGroupKFold(
            n_splits=n_splits, shuffle=True, random_state=current_random_state
        )
        
        for train_idx, val_idx in splitter.split(X, y, groups=groups):
            global_fold_index += 1
            
            # Dividir datos en entrenamiento y validación
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            
            # Entrenar modelo
            model.fit(X_train, y_train)
            
            # --- Métricas en conjunto de entrenamiento ---
            y_train_pred = model.predict(X_train)

            # Obtener probabilidades o scores de decisión si están disponibles
            if hasattr(model, "predict_proba"):
                y_train_prob = model.predict_proba(X_train)[:, 1]
            elif hasattr(model, "decision_function"):
                y_train_prob = model.decision_function(X_train)
            else:
                y_train_prob = None
            
            # Calcular AUC y F1 en entrenamiento
            try:
                train_auc = roc_auc_score(y_train, y_train_prob) if y_train_prob is not None else np.nan
            except:
                train_auc = np.nan
            train_f1 = f1_score(y_train, y_train_pred, average="binary")
            

            # --- Métricas en conjunto de validación ---
            y_val_pred = model.predict(X_val)
            
            # Obtener probabilidades para validación
            if hasattr(model, "predict_proba"):
                y_val_prob = model.predict_proba(X_val)[:, 1]
            elif hasattr(model, "decision_function"):
                y_val_prob = model.decision_function(X_val)
            else:
                y_val_prob = None
            
            # Calcular AUC en validación
            try:
                val_auc = roc_auc_score(y_val, y_val_prob) if y_val_prob is not None else np.nan
            except:
                val_auc = np.nan
            
            # Métricas de rendimiento completas en validación
            val_mcc = matthews_corrcoef(y_val, y_val_pred)          # Coeficiente de correlación Matthews
            val_kappa = cohen_kappa_score(y_val, y_val_pred)        # Kappa de Cohen (vs azar)
            val_f1_binary = f1_score(y_val, y_val_pred, average="binary")  # F1 binario
            val_f1_macro = f1_score(y_val, y_val_pred, average="macro")    # F1 macro
            val_accuracy = accuracy_score(y_val, y_val_pred)               # Accuracy
            val_balanced_accuracy = balanced_accuracy_score(y_val, y_val_pred)  # Accuracy balanceada
            val_sensitivity = recall_score(y_val, y_val_pred, pos_label=1)      # Sensibilidad
            val_specificity = recall_score(y_val, y_val_pred, pos_label=0)      # Especificidad
            val_ppv = precision_score(y_val, y_val_pred, pos_label=1)           # Valor predictivo positivo1)
            
            # Matriz de confusión para cálculos adicionales
            cm = confusion_matrix(y_val, y_val_pred)

            # Cálculo del valor predictivo negativo (NPV)
            if (cm[0, 0] + cm[1, 0]) > 0:
                val_npv = cm[0, 0] / (cm[0, 0] + cm[1, 0])
            else:
                val_npv = np.nan
            
            # Métricas por clase
            per_class_precision = precision_score(y_val, y_val_pred, average=None)
            per_class_recall = recall_score(y_val, y_val_pred, average=None)
            per_class_f1 = f1_score(y_val, y_val_pred, average=None)
            
            # Exactitud por clase (diagonal de la matriz normalizada por filas)
            per_class_accuracy = []
            for i in range(len(cm)):
                row_sum = np.sum(cm[i, :])
                if row_sum > 0:
                    per_class_accuracy.append(cm[i, i] / row_sum)
                else:
                    per_class_accuracy.append(np.nan)
            
             # Recopilar todas las métricas en un diccionario
            fold_metrics = {
                "Fold": global_fold_index,  
                "Repeat": rep + 1,          
                "train_auc": train_auc,
                "train_f1": train_f1,
                "val_auc": val_auc,
                "val_mcc": val_mcc,
                "val_kappa": val_kappa,
                "val_f1_binary": val_f1_binary,
                "val_f1_macro": val_f1_macro,
                "val_accuracy": val_accuracy,
                "val_sensitivity": val_sensitivity,
                "val_specificity": val_specificity,
                "val_ppv": val_ppv,
                "val_npv": val_npv,
                "val_balanced_accuracy": val_balanced_accuracy,
                "per_class_precision": per_class_precision.tolist(),
                "per_class_recall": per_class_recall.tolist(),
                "per_class_f1": per_class_f1.tolist(),
                "per_class_accuracy": per_class_accuracy
            }
            
            fold_results.append(fold_metrics)
    
            # Guardar datos de este fold para análisis posteriores (curvas ROC, etc.)
            folds_data.append({
                "fold_index": global_fold_index,
                "Repeat": rep + 1,
                "y_val": y_val,
                "y_val_pred": y_val_pred,
                "y_val_prob": y_val_prob 
            })
            
    pred_vals = {
        "folds": folds_data
    }
    return fold_results, pred_vals

# Código principal

In [4]:
"""
Función principal que coordina el proceso completo de entrenamiento y evaluación:
1. Procesa argumentos de línea de comandos
2. Carga y preprocesa los datos
3. Realiza selección de características (opcional)
4. Entrena y evalúa modelos
5. Genera curvas ROC y resultados
6. Ejecuta scripts complementarios (opcional)
"""
# --- Configuración de argumentos de línea de comandos ---    
parser = argparse.ArgumentParser(
    description="Evaluación de modelos con validación cruzada repetida"
)
parser.add_argument(
    "--csv", type=str,
    choices=["features_all_gland.csv", "features_all_full.csv"],
    default="features_all_gland.csv",
    help="Nombre del CSV con las características."
)
parser.add_argument(
    "--data_pre", type=str,
    default="../../../artifacts/radiomics",
    help="Directorio raíz donde se encuentran los datos radiomics."
)
parser.add_argument(
    "--results_base", type=str, default="../../../results/radiomics",
    help="Directorio base donde se crearán los resultados."
)
parser.add_argument(
    "--n_splits", type=int, default=5,
    help="Número de particiones para StratifiedGroupKFold (por repetición)."
)
parser.add_argument(
    "--n_repeats", type=int, default=10,
    help="Número de repeticiones de la validación cruzada."
)
parser.add_argument(
    "--feature_strategy", type=str,
    choices=["all", "most_discriminant"],
    default="most_discriminant",
    help="Estrategia de selección de features: 'all' o 'most_discriminant'."
)
parser.add_argument(
    "--calculate_differences", action="store_true", default=True,
    help="Si se habilita, ejecuta model_differences.py."
)
parser.add_argument(
    "--fine_tune_best_model", action="store_true", default=False,
    help="Si se habilita, realiza fine-tuning del mejor modelo."
)

args = parser.parse_args(args=[])



# Cargar datos

In [5]:
# --- Carga de datos y preprocesamiento ---
# label_csv= "label1" 
# num_label = label_csv[-1]
path_features = '/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/binary/features_t2w_MPfirrmann.csv'
df = pd.read_csv(path_features)
y = df["label"]
groups = df["patient_id"]
X = df.drop([ 'patient_id','study_id', 'label', 'mask_type',
                              'diagnostics_Versions_PyRadiomics', 'diagnostics_Versions_Numpy', 
                              'diagnostics_Versions_SimpleITK', 'diagnostics_Versions_PyWavelet', 
                              'diagnostics_Versions_Python', 'diagnostics_Configuration_Settings', 
                              'diagnostics_Configuration_EnabledImageTypes', 'diagnostics_Image-original_Hash', 
                              'diagnostics_Image-original_Dimensionality', 'diagnostics_Image-original_Spacing', 
                              'diagnostics_Image-original_Size', 'diagnostics_Image-original_Mean', 
                              'diagnostics_Image-original_Minimum', 'diagnostics_Image-original_Maximum', 
                              'diagnostics_Mask-original_Hash', 'diagnostics_Mask-original_Spacing', 
                              'diagnostics_Mask-original_Size', 'diagnostics_Mask-original_BoundingBox', 
                              'diagnostics_Mask-original_VoxelNum', 'diagnostics_Mask-original_VolumeNum', 
                              'diagnostics_Mask-original_CenterOfMassIndex', 'diagnostics_Mask-original_CenterOfMass', 
                              'diagnostics_Image-interpolated_Spacing', 'diagnostics_Image-interpolated_Size', 
                              'diagnostics_Image-interpolated_Mean', 'diagnostics_Image-interpolated_Minimum', 
                              'diagnostics_Image-interpolated_Maximum', 'diagnostics_Mask-interpolated_Spacing', 
                              'diagnostics_Mask-interpolated_Size', 'diagnostics_Mask-interpolated_BoundingBox', 
                              'diagnostics_Mask-interpolated_VoxelNum', 'diagnostics_Mask-interpolated_VolumeNum', 
                              'diagnostics_Mask-interpolated_CenterOfMassIndex', 'diagnostics_Mask-interpolated_CenterOfMass', 
                              'diagnostics_Mask-interpolated_Mean', 'diagnostics_Mask-interpolated_Minimum', 
                              'diagnostics_Mask-interpolated_Maximum'], axis=1)

experiment_dir = "/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass"
os.makedirs(experiment_dir, exist_ok=True)
print(f"Creada carpeta de resultados: {experiment_dir}")

Creada carpeta de resultados: /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass


In [9]:
# --- Selección de características ---
selected_features = X.columns

if args.feature_strategy == "most_discriminant":
    print(">> Realizando selección de características...")

    # Directorios para resultados de selección de características
    fs_dir = os.path.join(experiment_dir, "feature_selection")
    os.makedirs(fs_dir, exist_ok=True)
    images_dir = os.path.join(fs_dir, f"images")
    os.makedirs(images_dir, exist_ok=True)

    # Inicializar listas para almacenar estadísticas por característica
    feature_names, test_type_list, pvalue_list = ([] for _ in range(3))

    # Evaluar cada característica individualmente
    for column in X.columns:
        stat, p = shapiro(X[column])
        grupos = [X[column][y == clase] for clase in np.unique(y)]
        feature_names.append(column)
        alpha = 0.05
        if p > alpha:
            test_type_list.append('ANOVA')
            stats, pval = f_oneway(*grupos)
        else:
            test_type_list.append('Kruskal-Wallis')
            stats, pval = kruskal(*grupos)
        pvalue_list.append(pval)

    # Crear DataFrame con todas las estadísticas por característica
    train_auc_pvals_df = pd.DataFrame(
        list(zip(test_type_list, pvalue_list)),
        index=feature_names,
        columns=['Test', 'p-value']
    ).sort_values(by='p-value', ascending=True)

    # Seleccionar características: máximo 1 característica por cada 15 muestras
    num_features_model = round(X.shape[0] / 15)
    train_df = train_auc_pvals_df.sort_values(by='p-value', ascending=True)

    # Seleccionar las N características más significativas
    selected_features = train_df.index[0:num_features_model]
    print(f"  --> Seleccionadas {len(selected_features)} características más relevantes.")

    # Filtrar DataFrame para usar solo las características seleccionadas
    X = X[selected_features]
    # Guardar DataFrame con estadísticas completas
    df_path_1 = os.path.join(fs_dir, f"train_auc_pvals_df.csv")
    train_auc_pvals_df.loc[selected_features].to_csv(df_path_1)
    print(f"  --> Guardado CSV: {df_path_1}\n")


    # --- Generar visualizaciones para las TOP 20 características ---
    top_20 = train_auc_pvals_df.index[:20]

    for rank, feature_name in enumerate(top_20, start=1):
        # Crear nombre de archivo
        safe_feat_name = feature_name.replace("/", "_")
        feat_folder_name = f"{rank}_{safe_feat_name}"
        feat_folder_path = os.path.join(images_dir, feat_folder_name)
        os.makedirs(feat_folder_path, exist_ok=True)

        # 1. Gráfico de violín para visualizar distribuciones por clase
        plt.figure(figsize=(9, 9))
        sns.violinplot(x=y, y=df[feature_name], color='grey')
        plt.title(f"Distribución de {feature_name} por clase Pfirrmann", fontsize=14)
        plt.xlabel("Clase de Pfirrmann")
        plt.ylabel(feature_name)
        violin_plot_path = os.path.join(feat_folder_path, f"{safe_feat_name}_violinplot.png")
        plt.savefig(violin_plot_path, dpi=dpi)
        plt.close()
else:
    print(">> Usando TODAS las características (sin selección).")



>> Realizando selección de características...
  --> Seleccionadas 239 características más relevantes.
  --> Guardado CSV: /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass/feature_selection/train_auc_pvals_df.csv



# Entrenamiento y evaluación modelos

In [ ]:
#--- Entrenamiento y evaluación de modelos ---
models = get_models(random_state=42)

# Colectores para resultados
all_results = []
preds_data = []

# Evaluar cada modelo
for model_name, model in models:
    print(f"Evaluando {model_name}...")
    fold_metrics_list, pred_vals = evaluate_model_multiclass(
        model, X, y, groups,
        n_splits=args.n_splits,
        n_repeats=args.n_repeats,
        base_random_state=42
    )

    # Añadir nombre de clasificador a cada resultado
    for fold_metrics in fold_metrics_list:
        fold_metrics["Classifier"] = model_name
        all_results.append(fold_metrics)

    # Almacenar predicciones
    preds_data.append({
        "Classifier": model_name,
        "folds": pred_vals["folds"]
    })

# Crear DataFrame con todos los resultados
df_resultados = pd.DataFrame(all_results)

# Ordenar columnas para mejor legibilidad
fixed_cols = ["Classifier", "Fold", "Repeat"]
other_cols = [c for c in df_resultados.columns if c not in fixed_cols]
df_resultados = df_resultados[fixed_cols + other_cols]
df_resultados.sort_values(by=["Classifier", "Fold"], inplace=True)

# Generar nombre de archivo para resultados
resultados_filename = f"resultados_discoslumbar.csv"

#Guardar resultados
resultados_filepath = os.path.join(experiment_dir, resultados_filename)
df_resultados.to_csv(resultados_filepath, index=False)
print(f"\nResultados guardados en '{resultados_filepath}'")


# --- Estructurar datos de predicciones para guardar ---
records_for_csv = []
for item in preds_data:
    clf_name = item["Classifier"]
    folds_info = item["folds"]
    for fold_info in folds_info:
        fold_idx = fold_info["fold_index"]
        rep_idx = fold_info["Repeat"]
        
        y_val_list = fold_info["y_val"].tolist()
        y_pred_list = fold_info["y_val_pred"].tolist()
        if fold_info["y_val_prob"] is not None:
            y_prob_list = fold_info["y_val_prob"].tolist()
        else:
            y_prob_list = []
        
        records_for_csv.append({
            "Classifier": clf_name,
            "Fold": fold_idx,
            "Repeat": rep_idx,
            "y_val": y_val_list,
            "y_pred": y_pred_list,
            "y_prob": y_prob_list
        })

# Guardar predicciones en CSV
df_preds = pd.DataFrame(records_for_csv)
preds_filename = f"preds_discoslumbar.csv"
preds_filepath = os.path.join(experiment_dir, preds_filename)
df_preds.to_csv(preds_filepath, index=False)
print(f"Predicciones guardadas en '{preds_filepath}'")

# --- Guardar lista de variables utilizadas ---
variables_txt_path = os.path.join(experiment_dir, f"variables_usadas.txt")
with open(variables_txt_path, "w") as f:
    for feat in selected_features:
        f.write(str(feat) + "\n")
print(f"Archivo con variables usadas: {variables_txt_path}")





Evaluando SVM...
Evaluando Logistic Regression...
Evaluando Random Forest...


/mnt/datalake/openmind/MedP-Midas/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/mnt/datalake/openmind/MedP-Midas/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/mnt/datalake/openmind/MedP-Midas/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metr

Evaluando Naive Bayes...
Evaluando KNN...
Evaluando Gradient Boosting...

Resultados guardados en '/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass/resultados_discoslumbar.csv'
Predicciones guardadas en '/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass/preds_discoslumbar.csv'
Archivo con variables usadas: /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass/variables_usadas.txt


# ROC multiclase

In [3]:
from sklearn.preprocessing import label_binarize
import ast
import seaborn as sns

# --- Generación de curvas ROC MULTICLASE (One-vs-Rest) ---
print("\nGenerando curvas ROC multiclass (One-vs-Rest): fold óptimo y mediano por clasificador...")
experiment_dir = "/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass"


resultados_filename = f"resultados_discoslumbar.csv"
resultados_filepath = os.path.join(experiment_dir, resultados_filename)
df_resultados = pd.read_csv(resultados_filepath)

preds_filename = f"preds_discoslumbar.csv"
preds_filepath = os.path.join(experiment_dir, preds_filename)

df_preds = pd.read_csv(
    preds_filepath,
    converters={
      'y_val': ast.literal_eval,
      'y_prob': ast.literal_eval
    }
)


roc_dir = os.path.join(experiment_dir, f"ROC_curves")
os.makedirs(roc_dir, exist_ok=True)

curves_info_optimal = []
curves_info_median = []

# classifiers = df_resultados["Classifier"].unique()
classifiers = ['Gradient Boosting', 'SVM']

all_classes = np.unique(df_preds["y_val"].explode())  # obtiene todas las clases presentes
# all_classes = [1,2,3,4,5]
print(all_classes)

for clf_name in classifiers:
    df_clf = df_resultados[df_resultados["Classifier"] == clf_name]
    best_fold_idx = df_clf["val_auc"].idxmax()
    best_fold_num = df_clf.loc[best_fold_idx, "Fold"]
    median_auc = df_clf["val_auc"].median()
    median_fold_idx = (df_clf["val_auc"] - median_auc).abs().idxmin()
    median_fold_num = df_clf.loc[median_fold_idx, "Fold"]

    # Fold óptimo
    df_clf_preds_best = df_preds[
        (df_preds["Classifier"] == clf_name) & 
        (df_preds["Fold"] == best_fold_num)
    ]
    if len(df_clf_preds_best) > 0:
        y_val_list_best = df_clf_preds_best.iloc[0]["y_val"]
        y_prob_list_best = df_clf_preds_best.iloc[0]["y_prob"]
        if y_prob_list_best:
            y_val_bin = label_binarize(y_val_list_best, classes=all_classes)
            y_prob_arr = np.array(y_prob_list_best)
            fpr_dict, tpr_dict, auc_dict = {}, {}, {}
            for i, clase in enumerate(all_classes):
                fpr, tpr, _ = metrics.roc_curve(y_val_bin[:, i], y_prob_arr[:, i])
                auc_val = metrics.auc(fpr, tpr)
                fpr_dict[clase] = fpr
                tpr_dict[clase] = tpr
                auc_dict[clase] = auc_val
            curves_info_optimal.append({
                "classifier": clf_name,
                "fold": best_fold_num,
                "fpr": fpr_dict,
                "tpr": tpr_dict,
                "auc": auc_dict
            })

    # Fold mediano
    df_clf_preds_median = df_preds[
        (df_preds["Classifier"] == clf_name) & 
        (df_preds["Fold"] == median_fold_num)
    ]
    if len(df_clf_preds_median) > 0:
        y_val_list_median = df_clf_preds_median.iloc[0]["y_val"]
        y_prob_list_median = df_clf_preds_median.iloc[0]["y_prob"]
        if y_prob_list_median:
            y_val_bin = label_binarize(y_val_list_median, classes=all_classes)
            y_prob_arr = np.array(y_prob_list_median)
            fpr_dict, tpr_dict, auc_dict = {}, {}, {}
            for i, clase in enumerate(all_classes):
                fpr, tpr, _ = metrics.roc_curve(y_val_bin[:, i], y_prob_arr[:, i])
                auc_val = metrics.auc(fpr, tpr)
                fpr_dict[clase] = fpr
                tpr_dict[clase] = tpr
                auc_dict[clase] = auc_val
            curves_info_median.append({
                "classifier": clf_name,
                "fold": median_fold_num,
                "fpr": fpr_dict,
                "tpr": tpr_dict,
                "auc": auc_dict
            })

# --- Graficar ROC multiclass ---

# Número total de curvas = clasificadores * clases
n_curvas = len(curves_info_optimal) * len(all_classes)
palette = sns.color_palette("tab20", n_colors=n_curvas)

for info, fname in zip([curves_info_optimal, curves_info_median], ["roc_optimal_folds_multiclass.png", "roc_median_folds_multiclass.png"]):
    fig, ax = plt.subplots(figsize=(8, 6))
    color_idx = 0
    for clf_info in info:
        clf_name = clf_info["classifier"]
        fold_num = clf_info["fold"]
        auc_dict = clf_info["auc"]
        for clase in all_classes:
            fpr = clf_info["fpr"][clase]
            tpr = clf_info["tpr"][clase]
            auc_val = auc_dict[clase]
            color = palette[color_idx % len(palette)]
            ax.plot(fpr, tpr, label=f"{clf_name} (Fold={fold_num}, Clase={clase}, AUC={auc_val:.3f})", color=color)
            color_idx += 1
    ax.plot([0, 1], [0, 1], linestyle='--', color='gray', label="_nolegend_")
    ax.set_xlabel("False Positive Rate", fontsize=12, labelpad=10)
    ax.set_ylabel("True Positive Rate", fontsize=12, labelpad=10)
    ax.tick_params(axis='both', which='major', labelsize=10)
    ax.legend(fontsize=8)
    fig.tight_layout()
    #save pdf
    plt.savefig(os.path.join(roc_dir, fname.replace(".png", ".pdf")), dpi=dpi, bbox_inches='tight')
    #save png
    plt.savefig(os.path.join(roc_dir, fname), dpi=dpi, bbox_inches='tight')
    plt.close(fig)
    print(f"Gráfico ROC multiclass guardado en: {os.path.join(roc_dir, fname)}")


Generando curvas ROC multiclass (One-vs-Rest): fold óptimo y mediano por clasificador...
[1.0 2.0 3.0 4.0 5.0]
Gráfico ROC multiclass guardado en: /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass/ROC_curves/roc_optimal_folds_multiclass.png
Gráfico ROC multiclass guardado en: /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass/ROC_curves/roc_median_folds_multiclass.png


In [11]:
# Cargar el archivo CSV
df = pd.read_csv("/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass/resultados_discoslumbar.csv")

# Calcular el AUC promedio de validación por clasificador
mean_auc = df.groupby("Classifier")["val_auc"].mean()

# Mostrar el mejor modelo
best_model = mean_auc.idxmax()
best_auc = mean_auc.max()
print(f"Mejor modelo: {best_model} (AUC promedio: {best_auc:.3f})")
print("\nAUC promedio por modelo:")
print(mean_auc.sort_values(ascending=False))

Mejor modelo: SVM (AUC promedio: 0.877)

AUC promedio por modelo:
Classifier
SVM                    0.877027
Gradient Boosting      0.873190
Random Forest          0.864705
Logistic Regression    0.837932
Naive Bayes            0.806082
KNN                    0.778137
Name: val_auc, dtype: float64


In [16]:
# --- Ejecución de análisis adicionales (scripts complementarios) ---
resultados_filename = f"resultados_discoslumbar.csv"
resultados_filepath = os.path.join(experiment_dir, resultados_filename)
df_resultados = pd.read_csv(resultados_filepath)

preds_filename = f"preds_discoslumbar.csv"
preds_filepath = os.path.join(experiment_dir, preds_filename)
df_preds = pd.read_csv(preds_filepath)
print(resultados_filepath)

# Ejecutar script de comparación estadística de modelos (opcional)

print("\nEjecutando comparaciones de modelos (model_differences.py)...")
import subprocess

variables_txt_path = os.path.join(experiment_dir, f"variables_usadas.txt")

model_diff_dir = os.path.join(experiment_dir, f"model_differences")
os.makedirs(model_diff_dir, exist_ok=True)

# Construir comando para el script de comparación
postprocess_cmd = [
    "python3",
    "2_model_differences.py",
    "--csv_preds", preds_filepath,  # Archivo con predicciones
    "--csv_results", resultados_filepath,  # Archivo con métricas
    "--metric", "val_f1_macro",  # Métrica a comparar (AUC)
    "--alpha", "0.05",  # Nivel de significancia
    "--outdir", model_diff_dir  # Directorio de salida
]


/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass/resultados_discoslumbar.csv

Ejecutando comparaciones de modelos (model_differences.py)...


In [18]:
print(curves_info_optimal)

[{'classifier': 'Gradient Boosting', 'fold': 36, 'fpr': {1.0: array([0.        , 0.        , 0.00141643, 0.00141643, 0.00283286,
       0.00283286, 0.04957507, 0.04957507, 0.06232295, 0.06232295,
       0.08073654, 0.08073654, 0.08923513, 0.08923513, 0.12606232,
       0.12606232, 0.1529745 , 0.1529745 , 1.        ]), 2.0: array([0.        , 0.        , 0.00154799, 0.00154799, 0.01083591,
       0.01083591, 0.02012384, 0.02012384, 0.02167183, 0.02167183,
       0.0247678 , 0.0247678 , 0.02631579, 0.02631579, 0.02786378,
       0.02786378, 0.03250774, 0.03250774, 0.03560372, 0.03560372,
       0.03869969, 0.03869969, 0.04024768, 0.04024768, 0.04179567,
       0.04179567, 0.0495356 , 0.0495356 , 0.05263158, 0.05263158,
       0.05417957, 0.05417957, 0.05882353, 0.05882353, 0.06037152,
       0.06037152, 0.06346749, 0.06346749, 0.06501548, 0.06501548,
       0.06656347, 0.06656347, 0.06965944, 0.06965944, 0.07585139,
       0.07585139, 0.08049536, 0.08049536, 0.08513932, 0.08513932,
     

# SHAP multiclase (borrar/2)

In [31]:

for i in range(len(curves_info_optimal)):
    best_model = curves_info_optimal[i]["classifier"]
    print(best_model)


Gradient Boosting
SVM


In [ ]:
# Ejecutar script como proceso separado
subprocess.call(postprocess_cmd)


# Fine-tuning del mejor modelo (opcional)
if len(curves_info_optimal) > 0:
    # Identificar el mejor modelo (el primero en la lista ordenada)
    best_model = curves_info_optimal[0]["classifier"]

    # Mapeo de nombres para script de fine-tuning
    model_mapping = {
        "SVM": "SVM",
        "Logistic Regression": "LogisticRegression",
        "Random Forest": "RandomForest",
        "Naive Bayes": "NaiveBayes",
        "KNN": "KNN",
        "Gradient Boosting": "GradientBoosting"
    }
    best_model_finetune = model_mapping.get(best_model, best_model)
    
    print(f"Fine-tuning del mejor modelo: {best_model_finetune}")
    
    # Construir comando para script de fine-tuning
    fine_tune_cmd = [
        "python3",
        "3_retrain_best_model_and_evaluate_multiclass_copy.py",
        "--csv", path_features,                  # Mismo CSV de características
        "--model", best_model_finetune,     # Mejor modelo identificado
        "--variables", variables_txt_path   # Variables seleccionadas
    ]
    
    subprocess.call(fine_tune_cmd)
else:
    print("No se encontró información de curvas óptimas para determinar el mejor modelo.")


Ejecutando comparaciones de modelos (model_differences.py)...
  --> Resumen estadístico guardado en: /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass/model_differences/model_differences_summary.txt
  --> Boxplot guardado en: /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass/model_differences/boxplot_metric.png
  --> Heatmap de p-values guardado en: /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass/model_differences/heatmap_pvalues.png
Fine-tuning del mejor modelo: GradientBoosting

Iniciando fine-tuning del modelo.
  --> Modelo seleccionado: GradientBoosting
  --> CSV utilizado: /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/binary2/features_t2w_MPfirrmann.csv
  --> Archivo de variables: /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass/variables_usadas.txt

Carpeta de salida creada/ubicada en: ../

/mnt/datalake/openmind/MedP-Midas/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/mnt/datalake/openmind/MedP-Midas/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/mnt/datalake/openmind/MedP-Midas/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metr


Realizando análisis SHAP para entrenamiento...
 - Realizando test estadístico (Kruskal-Wallis) para entrenamiento con corrección Holm...
  > Procesando SHAP para la clase 0...
Error en SHAP analysis para entrenamiento: Shape of passed values is (200, 239), indices imply (2862, 239)

Realizando análisis SHAP para test...
 - Realizando test estadístico (Kruskal-Wallis) para test con corrección Holm...
  > Procesando SHAP para la clase 0...
Error en SHAP analysis para test: Shape of passed values is (200, 239), indices imply (718, 239)

Proceso finalizado. Report guardado en: ../data/features_t2w_multiclass/best_results/report.txt


In [7]:
# --- Ejecución de análisis adicionales (scripts complementarios) ---
resultados_filename = f"resultados_discoslumbar.csv"
resultados_filepath = os.path.join(experiment_dir, resultados_filename)
df_resultados = pd.read_csv(resultados_filepath)

preds_filename = f"preds_discoslumbar.csv"
preds_filepath = os.path.join(experiment_dir, preds_filename)
df_preds = pd.read_csv(preds_filepath)

# Ejecutar script de comparación estadística de modelos (opcional)

print("\nEjecutando comparaciones de modelos (model_differences.py)...")
import subprocess

variables_txt_path = os.path.join(experiment_dir, f"variables_usadas.txt")

model_diff_dir = os.path.join(experiment_dir, f"model_differences")
os.makedirs(model_diff_dir, exist_ok=True)
# os.mkdir(model_diff_dir)

# Construir comando para el script de comparación
postprocess_cmd = [
    "python3",
    "2_model_differences.py",
    "--csv_preds", preds_filepath,  # Archivo con predicciones
    "--csv_results", resultados_filepath,  # Archivo con métricas
    "--metric", "val_auc",  # Métrica a comparar (AUC)
    "--alpha", "0.05",  # Nivel de significancia
    "--outdir", model_diff_dir  # Directorio de salida
]

# # Ejecutar script como proceso separado
subprocess.call(postprocess_cmd)


# # Fine-tuning del mejor modelo (opcional)
# if len(curves_info_optimal) > 0:
#     # Identificar el mejor modelo (el primero en la lista ordenada)
#     best_model = curves_info_optimal[0]["classifier"]

#     # Mapeo de nombres para script de fine-tuning
#     model_mapping = {
#         "SVM": "SVM",
#         "Logistic Regression": "LogisticRegression",
#         "Random Forest": "RandomForest",
#         "Naive Bayes": "NaiveBayes",
#         "KNN": "KNN",
#         "Gradient Boosting": "GradientBoosting"
#     }
#     best_model_finetune = model_mapping.get(best_model, best_model)
    
#     print(f"Fine-tuning del mejor modelo: {best_model_finetune}")
    
#     # Construir comando para script de fine-tuning
#     fine_tune_cmd = [
#         "python3",
#         "3_retrain_best_model_and_evaluate_multiclass_copy.py",
#         "--csv", path_features,                  # Mismo CSV de características
#         "--model", best_model_finetune,     # Mejor modelo identificado
#         "--variables", variables_txt_path   # Variables seleccionadas
#     ]
    
#     subprocess.call(fine_tune_cmd)
# else:
#     print("No se encontró información de curvas óptimas para determinar el mejor modelo.")


Ejecutando comparaciones de modelos (model_differences.py)...
Iniciando análisis estadístico de modelos...
  --> Resumen estadístico guardado en: /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass/model_differences/model_differences_summary.txt
Classifier       SVM  Gradient Boosting  ...  Naive Bayes       KNN
Fold                                     ...                       
1           0.864077           0.862692  ...     0.795412  0.754975
2           0.889090           0.882793  ...     0.799484  0.791519
3           0.875838           0.881092  ...     0.818772  0.789755
4           0.882595           0.887806  ...     0.807924  0.796660
5           0.875243           0.859783  ...     0.811471  0.758348

[5 rows x 6 columns]
  --> Boxplot guardado en: /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass/model_differences/boxplot_metric.png
  --> Heatmap de p-values guardado en: /mnt/datalake/open

0